In [0]:
df_silver = spark.read.table("silver_credit")
print(f"Silver 筆數：{df_silver.count()}")
display(df_silver.limit(5))

In [0]:
from pyspark.sql import functions as F

df_gold = df_silver \
    .withColumn("loan_to_income_ratio",
        F.round(F.col("loan_amount") / F.col("income"), 4)
    ) \
    .withColumn("expense_to_income_ratio",
        F.round(F.col("monthly_expenses") * 12 / F.col("income"), 4)
    ) \
    .withColumn("savings_to_loan_ratio",
        F.round(F.col("savings_balance") / F.col("loan_amount"), 4)
    ) \
    .withColumn("financial_stress_score",
        F.round(
            F.col("debt_ratio") * 0.4 +
            F.col("credit_utilization") * 0.3 +
            (F.col("num_late_payments") / 20) * 0.3
        , 4)
    )

display(df_gold.limit(5))

In [0]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_credit_features")

print(f"Gold 寫入完成：{df_gold.count()} 筆")